# 01 · Country statistics: price levels, net earnings and exchange rates

**Owner:** Luca · **Step 1** (deadline 11 Nov 2026)

This notebook collects three country-level datasets via Web APIs. They are needed for the price comparisons (SQ3) and the *working hours per ski pass* index (SQ4):

| Data | Source | API |
| --- | --- | --- |
| Price level index (PLI), EU27 = 100 | Eurostat `prc_ppp_ind` | Eurostat dissemination API (via the `eurostat` Python package) |
| Annual net earnings of an average worker | Eurostat `earn_nt_netft` | same |
| EUR/CHF exchange rate, daily | Frankfurter (ECB reference rates) | REST API, no key |

Output: `data/processed/country_stats.csv` and `data/processed/fx_eur_chf_daily.csv`.

## 1. Setup

In [ ]:
import requests                 # sends requests to web APIs
import pandas as pd             # tables (DataFrames)
import matplotlib.pyplot as plt # charts
import eurostat                 # small library that talks to the Eurostat API
from pathlib import Path        # handles folder paths

COUNTRIES = ["CH", "IT", "AT", "FR"]   # ISO codes of the four countries
HOURS_PER_YEAR = 1760                  # assumption: 220 working days x 8 hours (same for all countries)

OUT = Path("../data/processed")        # where we save the results
OUT.mkdir(parents=True, exist_ok=True)

## 2. Price level index (Eurostat `prc_ppp_ind`)

- `na_item = PLI_EU27_2020`: price level index, EU27 average = 100 (e.g. 170 = prices 70 % above the EU average)
- `ppp_cat = A01`: *actual individual consumption*, i.e. the general cost of living

In [ ]:
pli_raw = eurostat.get_data_df(
    "prc_ppp_ind",
    filter_pars={"geo": COUNTRIES, "na_item": ["PLI_EU27_2020"], "ppp_cat": ["A01"]},
)
pli_raw.head()

In [ ]:
def wide_to_long(df, value_name):
    """Eurostat gives one column per year. Turn it into one row per country and year."""
    year_cols = [c for c in df.columns if str(c).strip()[:4].isdigit()]
    geo_col = [c for c in df.columns if str(c).startswith("geo")][0]
    long = df.melt(id_vars=[c for c in df.columns if c not in year_cols],
                   value_vars=year_cols, var_name="year", value_name=value_name)
    long = long.rename(columns={geo_col: "country"})
    long["year"] = long["year"].astype(str).str.strip().str[:4].astype(int)
    return long.dropna(subset=[value_name])

pli = wide_to_long(pli_raw, "pli_eu27")[["country", "year", "pli_eu27"]]
pli[pli["year"] >= 2019].pivot(index="year", columns="country", values="pli_eu27")

## 3. Net earnings (Eurostat `earn_nt_netft`)

We want: **net** earnings, **single person without children earning 100 % of the average wage**, in **EUR**.
The codes are looked up by their labels, and the full list is printed so that we can check the choice.

In [ ]:
ecase_labels = dict(eurostat.get_dic("earn_nt_netft", "ecase", full=False))
estruct_labels = dict(eurostat.get_dic("earn_nt_netft", "estruct", full=False))

print("ECASE options:")
for k, v in ecase_labels.items():
    print(" ", k, "-", v)
print("\nESTRUCT options:")
for k, v in estruct_labels.items():
    print(" ", k, "-", v)

In [ ]:
# pick the codes by their description
ECASE = [k for k, v in ecase_labels.items()
         if "without children" in v.lower() and "100%" in v.replace(" ", "")][0]
ESTRUCT = [k for k, v in estruct_labels.items() if v.lower().startswith("net earning")][0]
print("Chosen:", ECASE, "=", ecase_labels[ECASE])
print("Chosen:", ESTRUCT, "=", estruct_labels[ESTRUCT])
# If one of these lines gives an error: look at the printed lists above and write the right code by hand,
# e.g. ECASE = "P1_NCH_AW100"

In [ ]:
wage_raw = eurostat.get_data_df(
    "earn_nt_netft",
    filter_pars={"geo": COUNTRIES, "ecase": [ECASE], "estruct": [ESTRUCT], "currency": ["EUR"]},
)
wage = wide_to_long(wage_raw, "net_earnings_year_eur")[["country", "year", "net_earnings_year_eur"]]
wage["net_earnings_hour_eur"] = wage["net_earnings_year_eur"] / HOURS_PER_YEAR
wage[wage["year"] >= 2019].pivot(index="year", columns="country", values="net_earnings_year_eur").round(0)

## 4. EUR/CHF exchange rate (Frankfurter API)

In [ ]:
url = "https://api.frankfurter.dev/v1/2010-01-01..2025-12-31"
response = requests.get(url, params={"base": "EUR", "symbols": "CHF"}, timeout=60)
response.raise_for_status()          # stops here with an error message if the request failed
data = response.json()               # the answer is JSON: {"base": "EUR", "rates": {"2010-01-04": {"CHF": 1.48}, ...}}

fx = pd.DataFrame([{"date": d, "eur_chf": v["CHF"]} for d, v in data["rates"].items()])
fx["date"] = pd.to_datetime(fx["date"])
print(len(fx), "rows from", fx["date"].min().date(), "to", fx["date"].max().date())
fx.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(fx["date"], fx["eur_chf"], linewidth=1.2)
ax.set_title("The Swiss franc has gained strongly against the euro since 2010")
ax.set_ylabel("CHF per 1 EUR")
ax.set_xlabel("")
ax.grid(alpha=0.3)
fig.text(0.01, -0.02, "Source: ECB reference rates via Frankfurter API", fontsize=8)
plt.show()

## 5. Combine and save

`country_stats` = one row per country and year with price level and earnings. It will become a table in the SQLite database in step 3.

In [ ]:
country_stats = pli.merge(wage, on=["country", "year"], how="outer").sort_values(["country", "year"])

# integration check (needed for the integration log)
print("PLI rows:", len(pli), "| wage rows:", len(wage), "| merged rows:", len(country_stats))
print("Rows with both values:", country_stats.dropna().shape[0])

latest = country_stats.dropna().groupby("country")["year"].max()
print("\nLatest year with both values per country:")
print(latest)

country_stats.to_csv(OUT / "country_stats.csv", index=False)
fx.to_csv(OUT / "fx_eur_chf_daily.csv", index=False)
print("\nSaved:", OUT / "country_stats.csv", "and", OUT / "fx_eur_chf_daily.csv")

## 6. Quick check: what does one hour of work buy?

Latest common year. This is only a first look; the real index (SQ4) uses the scraped ski-pass prices.

In [ ]:
year = int(latest.min())
snap = country_stats[country_stats["year"] == year].set_index("country")
snap[["pli_eu27", "net_earnings_year_eur", "net_earnings_hour_eur"]].round(1)

## Notes for the integration log

- Join key: `country` (ISO code) + `year`.
- Record numbers lost/gained are printed in section 5.
- Assumption: 1,760 working hours per year in all four countries (real weekly hours differ, e.g. France 35 h). To be mentioned as a limitation.